# Build sum-of-products KEO for formaldehyde ($\mathrm{H_2CO}$) using user-defined valence coordinates

First, we compare the sum-of-products form with the numerical approach using small 1D grids to avoid generating large multidimensional representations that are needed only for comparison.

Then, we evaluate the sum-of-products form using larger 1D grids and store the results in a file.

In [1]:
import jax
import numpy as np
from jax import numpy as jnp
from scipy import constants

from keosop.gmat import Gmat
from keosop.hdf5 import save_gmat_hdf5
from keosop.numerical.keo import batch_Gmat, com, batch_pseudo
from keosop.revad import dihedral, dot, norm

jax.config.update("jax_enable_x64", True)

# converts G-matrix to cm^-1 units
G_to_invcm = (
    constants.value("Planck constant")
    * constants.value("Avogadro constant")
    * 1e16
    / (4.0 * np.pi**2 * constants.value("speed of light in vacuum"))
    * 1e5
)

Define the internal-to-Cartesian and Cartesian-to-internal coordinate transformations for $\mathrm{H_2CO}$.

The internal coordinates are defined as:

1. $r_{\mathrm{CO}}$ — C–O bond length,
2. $r_{\mathrm{CH_1}}$ — C–H1 bond length,
3. $r_{\mathrm{CH_2}}$ — C–H2 bond length,
4. $\alpha_{\mathrm{H_1CO}}$ — H1–C–O bond angle,
5. $\alpha_{\mathrm{H_2CO}}$ — H2–C–O bond angle,
6. $\Delta\tau = \tau + \pi$, where $\tau$ is the book angle (dihedral angle) between the (H1CO) and (H2CO) planes.

The atom ordering in Cartesian coordinates is:

$$
\mathrm{C},\ \mathrm{O},\ \mathrm{H_1},\ \mathrm{H_2}
$$

In [2]:
masses = np.array([12.0, 15.99491463, 1.00782505, 1.00782505])  # masses of C, O, H, H

ncoo = 6


def internal_to_cartesian(q):
    rCO, rCH1, rCH2, aOCH1, aOCH2, dtau = q
    tau = np.pi + dtau
    xyz = [
        [0.0, 0.0, 0.0],
        [0.0, 0.0, rCO],
        [
            rCH1 * np.sin(aOCH1) * np.cos(tau * 0.5),
            -rCH1 * np.sin(aOCH1) * np.sin(tau * 0.5),
            rCH1 * np.cos(aOCH1),
        ],
        [
            rCH2 * np.sin(aOCH2) * np.cos(tau * 0.5),
            rCH2 * np.sin(aOCH2) * np.sin(tau * 0.5),
            rCH2 * np.cos(aOCH2),
        ],
    ]
    return np.array(xyz, dtype=object)


def cartesian_to_internal(xyz):
    C, O, H1, H2 = xyz

    CO = O - C
    CH1 = H1 - C
    CH2 = H2 - C

    rCO = norm(CO)
    rCH1 = norm(CH1)
    rCH2 = norm(CH2)

    aOCH1 = np.arccos(dot(CO, CH1) / (rCO * rCH1))
    aOCH2 = np.arccos(dot(CO, CH2) / (rCO * rCH2))

    tau = dihedral(H1, C, O, H2)
    dtau = tau + np.pi

    return np.array([rCO, rCH1, rCH2, aOCH1, aOCH2, dtau], dtype=object)

Define one-dimensional grids for the internal coordinate representation and construct the G-matrix in sum-of-products form

In [3]:
rad = np.pi / 180

# define one-dimensional grids for each coordinate
grids = [
    np.linspace(0.8, 2.6, 5),  # r_{CO}
    np.linspace(0.8, 2.6, 5),  # r_{CH1}
    np.linspace(0.8, 2.6, 5),  # r_{CH2}
    np.linspace(30 * rad, 160 * rad, 5),  # alpha_{H1CO}
    np.linspace(30 * rad, 160 * rad, 5),  # alpha_{H2CO}
    np.linspace(-60 * rad, 60 * rad, 5),  # tau
]

ncoo = len(grids)

gmat = Gmat(grids, masses, internal_to_cartesian, cartesian_to_internal)
pseudo_sop = gmat.build_pseudopotential_fdf()

gvib_sop, grovib_sop, grot_sop = gmat.run(return_rot=True)

print("Number of terms for vibrational elements of G-matrix\n i  j no.terms")
for i in range(ncoo):
    for j in range(i, ncoo):
        print(f"{i:2d} {j:2d} {gvib_sop[i, j].nterms:6d}")

print("\nNumber of terms for rovibrational elements of G-matrix\n i  j no.terms")
for i in range(3):
    for j in range(ncoo):
        print(f"{i:2d} {j:2d} {grovib_sop[i, j].nterms:6d}")

print("\nNumber of terms for rotational elements of G-matrix\n i  j no.terms")
for i in range(3):
    for j in range(i, 3):
        print(f"{i:2d} {j:2d} {grot_sop[i, j].nterms:6d}")

print(f"Number of terms for pseudopotential: {pseudo_sop.nterms:6d}")

Number of terms for vibrational elements of G-matrix
 i  j no.terms
 0  0      1
 0  1      1
 0  2      1
 0  3      2
 0  4      2
 0  5      1
 1  1      1
 1  2      3
 1  3      3
 1  4      6
 1  5      2
 2  2      1
 2  3      6
 2  4      3
 2  5      2
 3  3      5
 3  4     12
 3  5      4
 4  4      5
 4  5      4
 5  5     14

Number of terms for rovibrational elements of G-matrix
 i  j no.terms
 0  0      1
 0  1      1
 0  2      1
 0  3      2
 0  4      2
 0  5      4
 1  0      1
 1  1      1
 1  2      1
 1  3      2
 1  4      2
 1  5      4
 2  0      1
 2  1      2
 2  2      2
 2  3      4
 2  4      4
 2  5      6

Number of terms for rotational elements of G-matrix
 i  j no.terms
 0  0      1
 0  1      1
 0  2      4
 1  1      1
 1  2      4
 2  2     14
Number of terms for pseudopotential:    142


Evaluate the G-matrix numerically on the multidimensional grid formed by the Cartesian product of the one-dimensional grids used to construct the sum-of-products representation

In [4]:
@com(masses)
def internal_to_cartesian_jax(q):
    rCO, rCH1, rCH2, aOCH1, aOCH2, dtau = q
    tau = jnp.pi + dtau
    xyz = [
        [0.0, 0.0, 0.0],
        [0.0, 0.0, rCO],
        [
            rCH1 * jnp.sin(aOCH1) * jnp.cos(tau * 0.5),
            -rCH1 * jnp.sin(aOCH1) * jnp.sin(tau * 0.5),
            rCH1 * jnp.cos(aOCH1),
        ],
        [
            rCH2 * jnp.sin(aOCH2) * jnp.cos(tau * 0.5),
            rCH2 * jnp.sin(aOCH2) * jnp.sin(tau * 0.5),
            rCH2 * jnp.cos(aOCH2),
        ],
    ]
    return jnp.array(xyz, dtype=jnp.float64)


# build direct-product grid
product_grid = np.stack(np.meshgrid(*grids, indexing="ij"), axis=-1)
product_grid = product_grid.reshape(-1, len(grids))

# map each multidimensional grid point to its 1D grid indices
grid_indices = np.indices([len(grid) for grid in grids]).reshape(len(grids), -1).T

# compute G-matrix on grid
gmat = batch_Gmat(product_grid, masses, internal_to_cartesian_jax)

# compute pseudopotential on grid
pseudo_num = batch_pseudo(product_grid, masses, internal_to_cartesian_jax)

gvib_num = gmat[:, :ncoo, :ncoo]
grovib_num = gmat[:, ncoo : ncoo + 3, :ncoo]
grot_num = gmat[:, ncoo : ncoo + 3, ncoo : ncoo + 3]

Calculate point-wise differences between the sum-of-products and numerical $G$-matrix

In [5]:
# evaluate one SOP term
def eval_sop_term(sop, iterm, inds):
    value = 1.0
    for k in range(sop.ncoo):
        value *= sop.terms[iterm][k][inds[k]]
    return value

# evaluate a complete SumProd object at one direct-product grid point
def eval_sop(sop, inds):
    return sum(
        eval_sop_term(sop, iterm, inds)
        for iterm in range(sop.nterms)
    )

In [6]:
# number of multidimensional grid points
npoints = product_grid.shape[0]

gvib_sop_eval = np.zeros((npoints, ncoo, ncoo), dtype=np.float64)
grot_sop_eval = np.zeros((npoints, 3, 3), dtype=np.float64)
grovib_sop_eval = np.zeros((npoints, 3, ncoo), dtype=np.float64)
pseudo_sop_eval = np.zeros((npoints,), dtype=np.float64)

# Gvib
for ipoint, inds in enumerate(grid_indices):
    for i in range(ncoo):
        for j in range(ncoo):
            gvib_sop_eval[ipoint, i, j] = eval_sop(gvib_sop[i, j], inds) * G_to_invcm


# Grot
for ipoint, inds in enumerate(grid_indices):
    for i in range(3):
        for j in range(3):
            grot_sop_eval[ipoint, i, j] = eval_sop(grot_sop[i, j], inds) * G_to_invcm


# Grovib
for ipoint, inds in enumerate(grid_indices):
    for i in range(3):
        for j in range(ncoo):
            grovib_sop_eval[ipoint, i, j] = (
                eval_sop(grovib_sop[i, j], inds) * G_to_invcm
            )

# pseudo
for ipoint, inds in enumerate(grid_indices):
    pseudo_sop_eval[ipoint] = eval_sop(pseudo_sop, inds) * G_to_invcm

In [7]:
# differences

diff_vib = gvib_sop_eval - gvib_num
diff_rovib = grovib_sop_eval - grovib_num
diff_rot = grot_sop_eval - grot_num
diff_pseudo = pseudo_sop_eval - pseudo_num

abs_diff_vib = np.abs(diff_vib)
abs_diff_rovib = np.abs(diff_rovib)
abs_diff_rot = np.abs(diff_rot)
abs_diff_pseudo = np.abs(diff_pseudo)

print("--- Gvib ---")
print("Maximum absolute difference:", np.max(abs_diff_vib))
print("Mean absolute difference:", np.mean(abs_diff_vib))
print("RMS difference:", np.sqrt(np.mean(diff_vib**2)))

print("\n--- Grovib ---")
print("Maximum absolute difference:", np.max(abs_diff_rovib))
print("Mean absolute difference:", np.mean(abs_diff_rovib))
print("RMS difference:", np.sqrt(np.mean(diff_rovib**2)))

print("\n--- Grot ---")
print("Maximum absolute difference:", np.max(abs_diff_rot))
print("Mean absolute difference:", np.mean(abs_diff_rot))
print("RMS difference:", np.sqrt(np.mean(diff_rot**2)))

print("\n--- Pseudo ---")
print("Maximum absolute difference:", np.max(abs_diff_pseudo))
print("Mean absolute difference:", np.mean(abs_diff_pseudo))
print("RMS difference:", np.sqrt(np.mean(diff_pseudo**2)))

--- Gvib ---
Maximum absolute difference: 9.322320693172514e-12
Mean absolute difference: 6.429313614397011e-15
RMS difference: 7.653459111047614e-14

--- Grovib ---
Maximum absolute difference: 5.5138116294983774e-12
Mean absolute difference: 1.0075871040063035e-14
RMS difference: 6.413750537245847e-14

--- Grot ---
Maximum absolute difference: 3.0411229090532288e-12
Mean absolute difference: 1.673276005398429e-14
RMS difference: 5.5404442609501556e-14

--- Pseudo ---
Maximum absolute difference: 7.983379572351623e-05
Mean absolute difference: 6.257286068652376e-06
RMS difference: 1.241815646851287e-05


In [8]:
print("--- Gvib element-wise errors ---")

for i in range(ncoo):
    for j in range(i, ncoo):
        d = diff_vib[:, i, j]
        max_abs = np.max(np.abs(d))
        rms = np.sqrt(np.mean(d**2))
        print(f"Gvib[{i},{j}]  " f"max abs = {max_abs:.3e}   " f"RMS = {rms:.3e}")

print("\n--- Grovib element-wise errors ---")

for i in range(3):
    for j in range(ncoo):
        d = diff_rovib[:, i, j]
        max_abs = np.max(np.abs(d))
        rms = np.sqrt(np.mean(d**2))
        print(f"Grovib[{i},{j}]  " f"max abs = {max_abs:.3e}   " f"RMS = {rms:.3e}")

print("\n--- Grot element-wise errors ---")

for i in range(3):
    for j in range(i, 3):
        d = diff_rot[:, i, j]
        max_abs = np.max(np.abs(d))
        rms = np.sqrt(np.mean(d**2))
        print(f"Grot[{i},{j}]  " f"max abs = {max_abs:.3e}   " f"RMS = {rms:.3e}")

--- Gvib element-wise errors ---
Gvib[0,0]  max abs = 2.665e-15   RMS = 1.821e-15
Gvib[0,1]  max abs = 1.998e-15   RMS = 4.045e-16
Gvib[0,2]  max abs = 2.442e-15   RMS = 5.627e-16
Gvib[0,3]  max abs = 3.997e-15   RMS = 7.618e-16
Gvib[0,4]  max abs = 3.997e-15   RMS = 8.499e-16
Gvib[0,5]  max abs = 1.815e-14   RMS = 8.107e-16
Gvib[1,1]  max abs = 4.974e-14   RMS = 9.711e-15
Gvib[1,2]  max abs = 2.309e-14   RMS = 2.499e-15
Gvib[1,3]  max abs = 3.209e-14   RMS = 7.126e-15
Gvib[1,4]  max abs = 3.642e-14   RMS = 2.819e-15
Gvib[1,5]  max abs = 2.256e-13   RMS = 8.883e-15
Gvib[2,2]  max abs = 3.553e-14   RMS = 1.158e-14
Gvib[2,3]  max abs = 3.419e-14   RMS = 2.708e-15
Gvib[2,4]  max abs = 3.642e-14   RMS = 7.015e-15
Gvib[2,5]  max abs = 2.043e-13   RMS = 8.704e-15
Gvib[3,3]  max abs = 9.948e-14   RMS = 1.348e-14
Gvib[3,4]  max abs = 4.796e-14   RMS = 4.218e-15
Gvib[3,5]  max abs = 4.050e-13   RMS = 1.624e-14
Gvib[4,4]  max abs = 8.527e-14   RMS = 1.354e-14
Gvib[4,5]  max abs = 5.462e-13   RMS

In [9]:
print("--- Gvib largest discrepancy ---")
idx = np.unravel_index(np.argmax(abs_diff_vib), abs_diff_vib.shape)
ipoint, i, j = idx

print("grid point :", ipoint)
print("coordinates:", np.asarray(product_grid[ipoint]))
print("element    :", (i, j))
print("SOP        :", gvib_sop_eval[ipoint, i, j])
print("reference  :", gvib_num[ipoint, i, j])
print("difference :", diff_vib[ipoint, i, j])

print("\n--- Grovib largest discrepancy ---")
idx = np.unravel_index(np.argmax(abs_diff_rovib), abs_diff_rovib.shape)
ipoint, i, j = idx

print("grid point :", ipoint)
print("coordinates:", np.asarray(product_grid[ipoint]))
print("element    :", (i, j))
print("SOP        :", grovib_sop_eval[ipoint, i, j])
print("reference  :", grovib_num[ipoint, i, j])
print("difference :", diff_rovib[ipoint, i, j])

print("\n--- Grot largest discrepancy ---")
idx = np.unravel_index(np.argmax(abs_diff_rot), abs_diff_rot.shape)
ipoint, i, j = idx

print("grid point :", ipoint)
print("coordinates:", np.asarray(product_grid[ipoint]))
print("element    :", (i, j))
print("SOP        :", grot_sop_eval[ipoint, i, j])
print("reference  :", grot_num[ipoint, i, j])
print("difference :", diff_rot[ipoint, i, j])

--- Gvib largest discrepancy ---
grid point : 2545
coordinates: [ 0.8         2.6         0.8         1.09083078  2.7925268  -1.04719755]
element    : (np.int64(5), np.int64(5))
SOP        : 611.1857196419018
reference  : 611.1857196419111
difference : -9.322320693172514e-12

--- Grovib largest discrepancy ---
grid point : 2545
coordinates: [ 0.8         2.6         0.8         1.09083078  2.7925268  -1.04719755]
element    : (np.int64(2), np.int64(5))
SOP        : 302.79546858980063
reference  : 302.79546858980615
difference : -5.5138116294983774e-12

--- Grot largest discrepancy ---
grid point : 2545
coordinates: [ 0.8         2.6         0.8         1.09083078  2.7925268  -1.04719755]
element    : (np.int64(2), np.int64(2))
SOP        : 157.31378202373958
reference  : 157.31378202374262
difference : -3.0411229090532288e-12


Now, generate the sum-of-products KEO using larger 1D grids and store the results in an HDF5 file

In [10]:
rad = np.pi / 180

# define one-dimensional grids for each coordinate
grids = [
    np.linspace(0.8, 2.6, 5000),  # r_{CO}
    np.linspace(0.8, 2.6, 5000),  # r_{CH1}
    np.linspace(0.8, 2.6, 5000),  # r_{CH2}
    np.linspace(30 * rad, 160 * rad, 5000),  # alpha_{H1CO}
    np.linspace(30 * rad, 160 * rad, 5000),  # alpha_{H2CO}
    np.linspace(-60 * rad, 60 * rad, 5000),  # tau
]

ncoo = len(grids)

# compute SOP for G-matrix
gmat = Gmat(grids, masses, internal_to_cartesian, cartesian_to_internal)

gvib_sop, grovib_sop, grot_sop = gmat.run(return_rot=True)

# store into file
filename = "keo_h2co_valence.h5"
save_gmat_hdf5(
    filename,
    grids,
    gvib_sop,
    grovib_sop,
    grot_sop,
    coo_names=("r_CO", "r_CH1", "r_CH2", "alpha_H1CO", "alpha_H2CO", "tau_H1COH2"),
    coo_units=("angs", "angs", "angs", "rad", "rad", "rad"),
)